In [1]:
from ultralytics import YOLO

import torch
import torchvision.transforms as transforms

from torchvision.models import resnet18
import torchvision.models as models
from PIL import Image

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from tqdm import tqdm
import time

In [2]:
YOLO_MODEL = YOLO("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\runs\\detect\\runs\\YOLOv8_baseline\\weights\\best.pt")

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = cnn.fc.in_features
cnn.fc = torch.nn.Sequential(
    torch.nn.Linear(num_features, 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(128, 2)
)

cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\best_chicken_cnn_augument.pth"))

cnn.to(DEVICE)

cnn.eval()

C:\Users\klanz\AppData\Local\Temp\ipykernel_28696\1784273219.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [5]:
def classify_crop(crop):

    image = Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB))

    tensor = transform(image)

    tensor = tensor.unsqueeze(0)

    tensor = tensor.to(DEVICE)

    with torch.no_grad():

        prediction = cnn(tensor)

        prediction = prediction.argmax(1).item()

    return prediction

In [6]:
def door_decision(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    cnn_classes = [p["cnn_prediction"] for p in predictions]

    if 1 in cnn_classes:
        return "CLOSE"

    if 0 in cnn_classes:
        return "OPEN"

    return "CLOSE"

In [7]:
def door_decision_yolo(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    classes = [p["class"] for p in predictions]

    if 1 in classes:
        return "CLOSE"

    if 0 in classes:
        return "OPEN"

    return "CLOSE"

In [8]:
CLASS_NAMES = {
    0: "chicken",
    1: "not_chicken"
}

In [9]:
def run_yolo(image_path, conf=0.5):

    result = YOLO_MODEL.predict(
        source=str(image_path),
        conf=conf,
        verbose=False
    )[0]

    predictions = []

    for box in result.boxes:

        cls = int(box.cls.item())

        confidence = float(box.conf.item())

        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]

        predictions.append({

            "class": cls,
            "class_name": CLASS_NAMES[cls],
            "confidence": confidence,
            "bbox": [int(x1), int(y1), int(x2), int(y2)]

        })

    return predictions

In [10]:
def run_pipeline(image_path, conf=0.5):

    detections = run_yolo(image_path, conf)

    image = cv2.imread(str(image_path))

    final_predictions = []

    for det in detections:

        x1, y1, x2, y2 = det["bbox"]

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        cnn_prediction = classify_crop(crop)

        final_predictions.append({

            "yolo_prediction": det["class"],

            "cnn_prediction": cnn_prediction,

            "confidence": det["confidence"],

            "bbox": det["bbox"]

        })

    return final_predictions

In [11]:
def load_ground_truth(label_path):

    if not Path(label_path).exists():
        return []

    gt=[]

    with open(label_path) as f:

        for line in f:

            line=line.strip()

            if line=="":

                continue

            cls=int(line.split()[0])

            gt.append(cls)

    return gt

In [12]:
def ground_truth_decision(gt_classes):

    if 1 in gt_classes:
        return "CLOSE"

    if 0 in gt_classes:
        return "OPEN"

    return "CLOSE"

In [13]:
test_images = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset\\test\\images").glob("*"))

results = []

In [14]:
for image_path in test_images:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")

    gt = load_ground_truth(label_path)

    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()

    yolo_predictions = run_yolo(image_path)

    yolo_time = (time.perf_counter()-start)*1000

    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()

    pipeline_predictions = run_pipeline(image_path)

    pipeline_time = (time.perf_counter()-start)*1000

    pipeline_decision = door_decision(pipeline_predictions)

    results.append({

        "image": image_path.name,

        "ground_truth": gt_decision,

        "yolo": yolo_decision,

        "pipeline": pipeline_decision,

        "yolo_time_ms": yolo_time,

        "pipeline_time_ms": pipeline_time,

        "objects_gt": gt,

        "objects_yolo": [x["class"] for x in yolo_predictions],

        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [15]:
df = pd.DataFrame(results)

df.head(20)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,445.7846,55.9645,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,12.4626,19.1003,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,19.4587,26.1895,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,11.6251,16.2175,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,19.7425,43.5004,"[0, 0, 0]","[0, 0, 0, 0]","[0, 0, 0, 1]"
5,1085.jpeg,OPEN,OPEN,CLOSE,19.1171,25.7094,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,14.1010,17.1266,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,16.4459,22.1601,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,13.4316,16.6153,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,11.7332,26.2691,"[0, 0, 0]","[0, 0]","[0, 0]"


In [16]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

import numpy as np

In [17]:
decision_map = {

    "OPEN":1,

    "CLOSE":0

}

In [18]:
def evaluate_system(df, prediction_column, time_column):

    gt = df["ground_truth"].map(decision_map)

    pred = df[prediction_column].map(decision_map)

    accuracy = accuracy_score(gt, pred)

    precision = precision_score(
        gt,
        pred,
        zero_division=0
    )

    recall = recall_score(
        gt,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        gt,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(gt, pred)

    tn, fp, fn, tp = cm.ravel()

    avg_time = df[time_column].mean()

    print("="*50)

    print(prediction_column)

    print("="*50)

    print(f"Accuracy : {accuracy:.4f}")

    print(f"Precision: {precision:.4f}")

    print(f"Recall   : {recall:.4f}")

    print(f"F1-score : {f1:.4f}")

    print()

    print(f"TP : {tp}")

    print(f"FP : {fp}")

    print(f"TN : {tn}")

    print(f"FN : {fn}")

    print()

    print(f"Średni czas: {avg_time:.2f} ms")

    return {

        "Accuracy":accuracy,

        "Precision":precision,

        "Recall":recall,

        "F1":f1,

        "TP":tp,

        "FP":fp,

        "TN":tn,

        "FN":fn,

        "Time":avg_time

    }

In [19]:
yolo_results = evaluate_system(

    df,

    "yolo",

    "yolo_time_ms"

)

yolo
Accuracy : 0.9886
Precision: 0.9958
Recall   : 0.9836
F1-score : 0.9897

TP : 479
FP : 2
TN : 389
FN : 8

Średni czas: 20.39 ms


In [20]:
pipeline_results = evaluate_system(

    df,

    "pipeline",

    "pipeline_time_ms"

)

pipeline
Accuracy : 0.9829
Precision: 0.9876
Recall   : 0.9815
F1-score : 0.9846

TP : 478
FP : 6
TN : 385
FN : 9

Średni czas: 28.63 ms


In [21]:
comparison = pd.DataFrame(

    [

        yolo_results,

        pipeline_results

    ],

    index=[

        "YOLO",

        "YOLO + CNN"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.988610,0.995842,0.983573,0.989669,479,2,389,8,20.390948
YOLO + CNN,0.982916,0.987603,0.981520,0.984552,478,6,385,9,28.625803


In [22]:
dangerous_yolo = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN")

]

In [23]:
dangerous_pipeline = df[

    (df["ground_truth"]=="CLOSE") &

    (df["pipeline"]=="OPEN")

]

In [24]:
print()

print("Krytyczne błędy")

print("----------------")

print("YOLO:",len(dangerous_yolo))

print("YOLO+CNN:",len(dangerous_pipeline))


Krytyczne błędy
----------------
YOLO: 2
YOLO+CNN: 6


In [25]:
improved = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN") &

    (df["pipeline"]=="CLOSE")

]

improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,15.2808,19.9173,[1],[0],[1]


In [26]:
empty_images = 0
false_detections = 0

for row in results:

    if len(row["objects_gt"]) == 0:

        empty_images += 1

        if len(row["objects_yolo"]) > 0:
            false_detections += 1

print("Puste obrazy:",empty_images)
print("Fałszywe detekcje:",false_detections)

if empty_images>0:

    print(
        "Odsetek:",
        false_detections/empty_images
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [27]:
fp_images=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["pipeline"]=="OPEN":

        fp_images.append(row["image"])

print("False Positive:",len(fp_images))

fp_images

False Positive: 6


['Image-107-dbb8ca.jpg',
 'Image-23-0a5765.jpg',
 'Image-88-8f30f6.jpg',
 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg',
 'raptor__raptor_043_jpg.rf.Zvj9EKGgySzsjUZnOEDi.jpg',
 'raptor__raptor_050_jpg.rf.duuf3tqrVk3ZzBLkisY3.jpg']

In [28]:
fn_images=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["pipeline"]=="CLOSE":

        fn_images.append(row["image"])

print("False Negative:",len(fn_images))

fn_images

False Negative: 9


['1054.jpeg',
 '1085.jpeg',
 'neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-FY33flUTFsva0xhQAxhZwgHaEK.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg',
 'OIP-tX7YgAqzRxQxMYkGe_FCtwHaFj.jpeg',
 'OIP-z_Y26rFvbazALq6sfXjgxwHaE8.jpeg']

In [29]:
fp_yolo=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["yolo"]=="OPEN":

        fp_yolo.append(row["image"])

len(fp_yolo)

fp_yolo

['Image-84-bd2f1b.jpg', 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [30]:
fn_yolo=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["yolo"]=="CLOSE":

        fn_yolo.append(row["image"])

len(fn_yolo)

fn_yolo

['neg_poultry__poultry_222_jpg.rf.g0AgDML3T5PV9uIHp0Xd.jpg',
 'neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-FY33flUTFsva0xhQAxhZwgHaEK.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg',
 'OIP-tX7YgAqzRxQxMYkGe_FCtwHaFj.jpeg',
 'OIP-z_Y26rFvbazALq6sfXjgxwHaE8.jpeg']

In [31]:

OUTPUT = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset_experiments")



In [32]:
image_path_dark = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\dark\\images").glob("*"))
image_path_night = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\night\\images").glob("*"))
image_path_occlusion = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\occlusion\\images").glob("*"))
image_path_motion_blur = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\motion_blur\\images").glob("*"))

results_dark = []
results_night = []
results_occlusion = []
results_motion_blur = []


In [33]:
for image_path in image_path_dark:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_dark.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [34]:
for image_path in image_path_night:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_night.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [35]:
for image_path in image_path_occlusion:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_occlusion.append({
        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [36]:
for image_path in image_path_motion_blur:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_motion_blur.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [37]:
df_dark = pd.DataFrame(results_dark)
df_night = pd.DataFrame(results_night)
df_occlusion = pd.DataFrame(results_occlusion)
df_motion_blur = pd.DataFrame(results_motion_blur)

df_dark.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,342.6079,29.3996,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,25.3783,29.1209,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,22.8842,32.7811,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,22.6453,23.2929,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,18.9184,33.0287,"[0, 0, 0]","[0, 0, 0]","[0, 1, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,18.6388,25.6455,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,17.7916,21.5820,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,18.1662,24.2174,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,18.3830,23.0350,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,19.6142,31.0309,"[0, 0, 0]","[0, 0]","[0, 0]"


In [38]:
df_night.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,31.1923,22.5518,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,19.1290,24.0732,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,18.5965,33.9970,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,28.6177,23.0632,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,OPEN,18.0870,38.0709,"[0, 0, 0]","[0, 0, 0, 0]","[0, 0, 0, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,33.8429,21.8767,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,18.6867,23.8006,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,22.4252,27.7336,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,17.9829,23.8268,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,19.6539,27.4681,"[0, 0, 0]","[0, 0]","[0, 0]"


In [39]:
df_occlusion.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,23.7009,12.9176,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,14.1852,18.2758,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,13.2401,21.8752,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,14.6553,21.8278,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,14.9795,35.8160,"[0, 0, 0]","[0, 0, 0, 0]","[0, 0, 1, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,28.9151,25.5317,[0],"[0, 0]","[1, 0]"
6,109.jpeg,OPEN,OPEN,OPEN,19.2760,22.7060,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,17.8608,18.5483,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,17.0347,22.7558,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,24.0772,26.5625,"[0, 0, 0]","[0, 0]","[0, 0]"


In [40]:
df_motion_blur.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,18.4078,23.8834,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,18.8587,24.7953,[0],[0],[0]
2,1048.jpeg,OPEN,CLOSE,CLOSE,18.6272,17.3167,[0],[],[]
3,1053.jpeg,OPEN,OPEN,OPEN,18.4223,23.7949,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,CLOSE,CLOSE,18.9177,22.4531,"[0, 0, 0]",[1],[1]
5,1085.jpeg,OPEN,OPEN,CLOSE,19.1496,23.6873,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,19.4876,23.6756,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,18.3461,22.9648,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,19.2056,23.6646,[0],[0],[0]
9,1133.jpeg,OPEN,CLOSE,CLOSE,17.5153,18.2169,"[0, 0, 0]",[],[]


In [41]:
yolo_results1 = evaluate_system(
    df_dark,
    "yolo",
    "yolo_time_ms"
)
pipeline_results1 = evaluate_system(
    df_dark,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9829
Precision: 0.9876
Recall   : 0.9815
F1-score : 0.9846

TP : 478
FP : 6
TN : 385
FN : 9

Średni czas: 19.63 ms
pipeline
Accuracy : 0.9784
Precision: 0.9875
Recall   : 0.9733
F1-score : 0.9804

TP : 474
FP : 6
TN : 385
FN : 13

Średni czas: 26.91 ms


In [42]:
yolo_results2 = evaluate_system(
    df_night,
    "yolo",
    "yolo_time_ms"
)
pipeline_results2 = evaluate_system(
    df_night,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9658
Precision: 0.9831
Recall   : 0.9548
F1-score : 0.9688

TP : 465
FP : 8
TN : 383
FN : 22

Średni czas: 20.36 ms
pipeline
Accuracy : 0.9636
Precision: 0.9830
Recall   : 0.9507
F1-score : 0.9666

TP : 463
FP : 8
TN : 383
FN : 24

Średni czas: 26.62 ms


In [43]:
yolo_results3 = evaluate_system(
    df_occlusion,
    "yolo",
    "yolo_time_ms"
)
pipeline_results3 = evaluate_system(
    df_occlusion,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9590
Precision: 0.9913
Recall   : 0.9343
F1-score : 0.9619

TP : 455
FP : 4
TN : 387
FN : 32

Średni czas: 19.95 ms
pipeline
Accuracy : 0.9487
Precision: 0.9804
Recall   : 0.9261
F1-score : 0.9525

TP : 451
FP : 9
TN : 382
FN : 36

Średni czas: 26.70 ms


In [44]:
yolo_results4 = evaluate_system(
    df_motion_blur,
    "yolo",
    "yolo_time_ms"
)
pipeline_results4 = evaluate_system(
    df_motion_blur,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.8189
Precision: 0.9227
Recall   : 0.7351
F1-score : 0.8183

TP : 358
FP : 30
TN : 361
FN : 129

Średni czas: 19.10 ms
pipeline
Accuracy : 0.7882
Precision: 0.9688
Recall   : 0.6386
F1-score : 0.7698

TP : 311
FP : 10
TN : 381
FN : 176

Średni czas: 23.28 ms


In [45]:
comparison = pd.DataFrame(
    [
        yolo_results1,
        pipeline_results1
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.982916,0.987603,0.981520,0.984552,478,6,385,9,19.629199
YOLO + CNN,0.978360,0.987500,0.973306,0.980352,474,6,385,13,26.908578


In [46]:
comparison = pd.DataFrame(
    [
        yolo_results2,
        pipeline_results2

    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.965831,0.983087,0.954825,0.968750,465,8,383,22,20.358071
YOLO + CNN,0.963554,0.983015,0.950719,0.966597,463,8,383,24,26.622184


In [47]:
comparison = pd.DataFrame(
    [
        yolo_results3,
        pipeline_results3
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.958998,0.991285,0.934292,0.961945,455,4,387,32,19.947825
YOLO + CNN,0.948747,0.980435,0.926078,0.952482,451,9,382,36,26.695037


In [48]:
comparison = pd.DataFrame(
    [
        yolo_results4,
        pipeline_results4
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.818907,0.922680,0.735113,0.818286,358,30,361,129,19.097898
YOLO + CNN,0.788155,0.968847,0.638604,0.769802,311,10,381,176,23.278321


In [49]:
dangerous_yolo1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN")
]

In [50]:
dangerous_yolo2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN")
]

In [51]:
dangerous_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN")
]

In [52]:
dangerous_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN")
]

In [53]:
dangerous_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["pipeline"]=="OPEN")
]

In [54]:
dangerous_pipeline2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["pipeline"]=="OPEN")
]

In [55]:
dangerous_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["pipeline"]=="OPEN")
]

In [56]:
dangerous_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]

In [69]:
dangerous_both1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") & (df_dark["yolo"]=="OPEN") & (df_dark["pipeline"]=="OPEN")
]
dangerous_both2 = df_night[
    (df_night["ground_truth"]=="CLOSE") & (df_night["yolo"]=="OPEN") & (df_night["pipeline"]=="OPEN")
]
dangerous_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") & (df_occlusion["yolo"]=="OPEN") & (df_occlusion["pipeline"]=="OPEN")
]
dangerous_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") & (df_motion_blur["yolo"]=="OPEN") & (df_motion_blur["pipeline"]=="OPEN")
]


In [68]:
comparison = pd.DataFrame(

    [   

        yolo_results,

        pipeline_results,

        yolo_results1,

        pipeline_results1,

        yolo_results2,

        pipeline_results2,

        yolo_results3,

        pipeline_results3,

        yolo_results4,

        pipeline_results4

    ],

    index=[

        "YOLO normal",

        "YOLO + CNN normal",

        "YOLO dark",

        "YOLO + CNN dark",

        "YOLO night",

        "YOLO + CNN night",

        "YOLO occlusion",

        "YOLO + CNN occlusion",

        "YOLO motion",

        "YOLO + CNN motion"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO normal,0.988610,0.995842,0.983573,0.989669,479,2,389,8,20.390948
YOLO + CNN normal,0.982916,0.987603,0.981520,0.984552,478,6,385,9,28.625803
YOLO dark,0.982916,0.987603,0.981520,0.984552,478,6,385,9,19.629199
YOLO + CNN dark,0.978360,0.987500,0.973306,0.980352,474,6,385,13,26.908578
YOLO night,0.965831,0.983087,0.954825,0.968750,465,8,383,22,20.358071
YOLO + CNN night,0.963554,0.983015,0.950719,0.966597,463,8,383,24,26.622184
YOLO occlusion,0.958998,0.991285,0.934292,0.961945,455,4,387,32,19.947825
YOLO + CNN occlusion,0.948747,0.980435,0.926078,0.952482,451,9,382,36,26.695037
YOLO motion,0.818907,0.922680,0.735113,0.818286,358,30,361,129,19.097898
YOLO + CNN motion,0.788155,0.968847,0.638604,0.769802,311,10,381,176,23.278321


In [70]:
print()
print("Krytyczne błędy, wpuszczenie drapieżnika")
print("----------------")
print("YOLO dark:",len(dangerous_yolo1),"     YOLO night:",len(dangerous_yolo2),"     YOLO occlusion:",len(dangerous_yolo3),"    YOLO motion:",len(dangerous_yolo4))
print("YOLO+CNN dark:",len(dangerous_pipeline1),"YOLO+CNN night:",len(dangerous_pipeline2),"YOLO+CNN occlusion:",len(dangerous_pipeline3),"YOLO+CNN motion:",len(dangerous_pipeline4))
print("BOTH dark:",len(dangerous_both1),"     BOTH night:",len(dangerous_both2),"     BOTH occlusion:",len(dangerous_both3),"    BOTH motion:",len(dangerous_both4))


Krytyczne błędy, wpuszczenie drapieżnika
----------------
YOLO dark: 6      YOLO night: 8      YOLO occlusion: 4     YOLO motion: 30
YOLO+CNN dark: 6 YOLO+CNN night: 8 YOLO+CNN occlusion: 9 YOLO+CNN motion: 10
BOTH dark: 3      BOTH night: 4      BOTH occlusion: 2     BOTH motion: 3


In [71]:
locking_chicken1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE") | (df_dark["pipeline"]=="CLOSE"))
]
locking_chicken2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE") | (df_night["pipeline"]=="CLOSE"))
]
locking_chicken3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE") | (df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE") | (df_motion_blur["pipeline"]=="CLOSE"))
]

In [ ]:
print()
print("Niekrytyczne błędy, niewpuszczenie kur (both)")
print("----------------")
print("YOLO dark:",len(locking_chicken1),"     YOLO night:",len(locking_chicken2),"     YOLO occlusion:",len(locking_chicken3),"    YOLO motion:",len(locking_chicken4))



Niekrytyczne błędy, niewpuszczenie kur
----------------
YOLO dark: 14      YOLO night: 26      YOLO occlusion: 36     YOLO motion: 181


In [58]:
improved = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN") &
    (df_dark["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,20.8625,22.0396,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,16.6425,24.3419,[1],"[0, 0]","[1, 1]"
833,raptor__gbif_raptor_00646_jpg.rf.IC8W8N97wC0S3...,CLOSE,OPEN,CLOSE,17.0450,19.6706,[1],[0],[1]


In [59]:
improved = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN") &
    (df_night["pipeline"]=="CLOSE")

]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,28.2343,24.1478,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,13.8237,22.2406,[1],"[0, 0]","[1, 1]"
836,raptor__gbif_raptor_00679_jpg.rf.IO9lWEg1PBD8V...,CLOSE,OPEN,CLOSE,18.0851,23.2205,[1],[0],[1]
840,raptor__gbif_raptor_00731_jpg.rf.DaBUw9KxmsnkZ...,CLOSE,OPEN,CLOSE,14.5598,18.0400,[1],[0],[1]


In [60]:
improved = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN") &
    (df_occlusion["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,29.5434,22.1959,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,15.7251,19.9530,[1],[0],[1]


In [61]:
improved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN") &
    (df_motion_blur["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
166,fox__gbif_fox_0058_jpg.rf.276TrYjTZ8l8fHW2cbDV...,CLOSE,OPEN,CLOSE,15.7131,20.2259,[1],[0],[1]
193,fox__gbif_fox_0540_jpg.rf.SviRhT8VBkdRDGFU1QO6...,CLOSE,OPEN,CLOSE,15.1898,19.8352,[1],[0],[1]
196,fox__gbif_fox_0615_jpg.rf.xGBhUyQrn1rpiysNE7WH...,CLOSE,OPEN,CLOSE,14.8426,19.8869,[1],[0],[1]
198,fox__gbif_fox_0722_jpg.rf.4EaY143B8mKPFC0058JJ...,CLOSE,OPEN,CLOSE,16.2510,22.5715,[1],[0],[1]
226,Image-111-ed55ed.jpg,CLOSE,OPEN,CLOSE,28.5972,20.2723,[1],[0],[1]
246,Image-28-e7f0ae.jpg,CLOSE,OPEN,CLOSE,26.0001,19.0395,"[1, 1]",[0],[1]
250,Image-31-1d307b.jpg,CLOSE,OPEN,CLOSE,26.1899,20.8673,[1],[0],[1]
252,Image-32-08b1fa.jpg,CLOSE,OPEN,CLOSE,15.6636,18.7688,[1],[0],[1]
266,Image-43-ac927e.jpg,CLOSE,OPEN,CLOSE,28.8059,18.8249,"[1, 1]",[0],[1]
274,Image-48-7fc754.jpg,CLOSE,OPEN,CLOSE,30.6138,23.9578,[1],[0],[1]


In [62]:
deproved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]
deproved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
122,coyote__lila_Felidae_Conservation_Fund_2020-20...,CLOSE,CLOSE,OPEN,14.7185,19.8462,[1],[1],[0]
202,fox__lila_AMMonitor_Camera_Traps_Kels5_0219_20...,CLOSE,CLOSE,OPEN,16.0254,20.7580,[1],[1],[0]
220,Image-107-dbb8ca.jpg,CLOSE,CLOSE,OPEN,21.2876,30.4279,[1],[1],[0]
264,Image-42-dd926b.jpg,CLOSE,CLOSE,OPEN,26.7442,46.5838,[1],[1],[0]
318,Image-78-0546ac.jpg,CLOSE,CLOSE,OPEN,21.2484,34.5890,[1],"[1, 0]","[0, 0]"
355,neg_empty__background_empty_043_jpg.rf.XPpXqOG...,CLOSE,CLOSE,OPEN,19.0969,26.7890,[],[1],[0]
874,raptor__raptor_027_jpg.rf.PzIAqRQVSQEonrCHFVSD...,CLOSE,CLOSE,OPEN,20.4718,27.6271,"[1, 1]",[1],[0]


In [63]:
empty_images0 = 0
false_detections0 = 0
for row in results_dark:

    if len(row["objects_gt"]) == 0:

        empty_images0 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections0 += 1

print("Puste obrazy:",empty_images0)
print("Fałszywe detekcje:",false_detections0)
if empty_images0>0:

    print(
        "Odsetek:",
        false_detections0/empty_images0
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [64]:
empty_images1 = 0
false_detections1 = 0

for row in results_night:

    if len(row["objects_gt"]) == 0:

        empty_images1 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections1 += 1

print("Puste obrazy:",empty_images1)
print("Fałszywe detekcje:",false_detections1)

if empty_images1>0:

    print(
        "Odsetek:",
        false_detections1/empty_images1
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [65]:
empty_images2 = 0
false_detections2 = 0

for row in results_occlusion:

    if len(row["objects_gt"]) == 0:

        empty_images2 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections2 += 1

print("Puste obrazy:",empty_images2)
print("Fałszywe detekcje:",false_detections2)

if empty_images2>0:

    print(
        "Odsetek:",
        false_detections2/empty_images2
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [66]:
empty_images3 = 0
false_detections3 = 0

for row in results_motion_blur:

    if len(row["objects_gt"]) == 0:

        empty_images3 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections3 += 1

print("Puste obrazy:",empty_images3)
print("Fałszywe detekcje:",false_detections3)

if empty_images3>0:

    print(
        "Odsetek:",
        false_detections3/empty_images3
    )

Puste obrazy: 37
Fałszywe detekcje: 4
Odsetek: 0.10810810810810811


In [67]:

#ZMIEŃ CONFIDENC POTEM NA 0,5 I PORÓWNAJ!!!!!!